# Error analysis in thin notebook

# inspect

In [ ]:
# ---- Section: Imports and constants ----
import numpy as np  # noqa: F401
import pandas as pd

from digikala_review_transformer import config

RUNS_ROOT = config.ROOT / "artifacts" / "transformer_runs"
RUN_NAMES = ["tooka_B", "tooka_C", "mdeberta_B", "roberta_B"]
PACK_DIR = config.ROOT / "data" / "training_pack"
BASE_DIR = config.ROOT / "reports" / "predictions"
OUT_DIR = config.ROOT / "reports" / "error_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TFIDF_VARIANT = "B"   # "A" or "B"; used for the TF-IDF column in the tables

HEADS = {
    "rec": {
        "cols": ["p_recommended", "p_not_recommended", "p_no_idea"],
        "y": "y_rec", "y_raw": "y_rec_raw",
        "tfidf": f"tfidf12_rec_{TFIDF_VARIANT}__test.parquet",
    },
    "band": {
        "cols": ["p_low", "p_medium", "p_high"],
        "y": "y_band", "y_raw": "y_band_raw",
        "tfidf": f"tfidf12_band_{TFIDF_VARIANT}__test.parquet",
    },
}

# ---- Section: Test pack (ids, body, labels) ----
test = pd.read_parquet(
    PACK_DIR / "test.parquet",
    columns=["id", "body", "y_rec", "y_band", "y_rec_raw", "y_band_raw"],
)
test["id"] = test["id"].astype(str)
print(f"test rows: {len(test):,}")


# ---- Section: Helpers ----
def load_pred(path, cols, ref_ids):
    """Read a probs parquet -> (pred_idx Series, conf Series) indexed by id."""
    df = pd.read_parquet(path)
    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
        df = df.drop_duplicates("id").set_index("id")
    else:
        assert len(df) == len(ref_ids), f"{path.name}: no 'id' and length mismatch"
        df.index = pd.Index(ref_ids)
    P = df[cols].to_numpy(float)
    return (pd.Series(P.argmax(1), index=df.index),
            pd.Series(P.max(1), index=df.index))


# ---- Section: Build one wide table per head ----
for head, h in HEADS.items():
    names = [c.removeprefix("p_") for c in h["cols"]]
    name_of = dict(enumerate(names))

    base = test[["id", "body", h["y"], h["y_raw"]]].rename(
        columns={h["y"]: "y_true", h["y_raw"]: "y_true_raw"}
    )
    base = base[base["y_true"].isin([0, 1, 2])].copy()   # valid targets only
    base["y_true"] = base["y_true"].astype(int)
    base = base.set_index("id")

    models = {"tfidf": BASE_DIR / h["tfidf"]}
    models.update({r: RUNS_ROOT / r / "test_probs.parquet" for r in RUN_NAMES})

    wrong = {}
    for m, path in models.items():
        pred, conf = load_pred(path, h["cols"], test["id"].to_numpy())
        pred, conf = pred.reindex(base.index), conf.reindex(base.index)
        assert pred.notna().all(), f"{head}/{m}: missing ids in predictions"
        pred = pred.astype(int)
        base[f"{m}_pred"] = pred.map(name_of)
        base[f"{m}_conf"] = conf.round(4)
        wrong[m] = (pred != base["y_true"]).to_numpy()

    W = pd.DataFrame(wrong, index=base.index)
    base["y_true"] = base["y_true"].map(name_of)
    base["n_wrong"] = W.sum(axis=1)
    base["n_correct"] = len(models) - base["n_wrong"]
    base["wrong_models"] = W.apply(lambda r: ",".join(r.index[r]), axis=1)

    out = (base[base["n_wrong"] >= 1]
           .reset_index()
           .sort_values(["n_wrong", "id"], ascending=[False, True]))

    path_out = OUT_DIR / f"errors_{head}.csv"
    out.to_csv(path_out, index=False, encoding="utf-8-sig")

    # ---- quick summary ----
    print(f"\n=== {head} === test valid: {len(base):,} | rows with >=1 error: {len(out):,} "
          f"| all {len(models)} wrong: {(out['n_wrong'] == len(models)).sum():,}")
    summ = pd.DataFrame({
        "errors": W.sum(),
        "only_this_wrong": [(W[m] & (W.sum(axis=1) == 1)).sum() for m in W.columns],
    })
    print(summ)
    print("saved ->", path_out)

test rows: 922,805

=== rec === test valid: 787,845 | rows with >=1 error: 186,130 | all 5 wrong: 86,495
            errors  only_this_wrong
tfidf       144267            25734
tooka_B     124614             3087
tooka_C     125455             2182
mdeberta_B  128472             5358
roberta_B   127562             5270
saved -> /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/reports/error_analysis/errors_rec.csv

=== band === test valid: 922,804 | rows with >=1 error: 374,211 | all 5 wrong: 195,775
            errors  only_this_wrong
tfidf       299696            44673
tooka_B     275170             5564
tooka_C     274243             3880
mdeberta_B  262259             6800
roberta_B   275651             8278
saved -> /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/reports/error_analysis/errors_band.csv


# importing HTML maker script

In [1]:
from digikala_review_transformer.errors_explorer import build_errors_explorer

build_errors_explorer()          # -> reports/error_analysis/errors_explorer.html

saved -> /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/reports/error_analysis/errors_explorer.html | 97.2 MB | rows: {'rec': 186130, 'band': 374211}


PosixPath('/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/reports/error_analysis/errors_explorer.html')